In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import cross_val_score
from itertools import product  


In [10]:
from main import load_data
from main import check_missing ,drop_threshold ,cross_val_splits

path='data/Womens Clothing E-Commerce Reviews.csv'
df=load_data(path)
df.head()

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [3]:
report=check_missing(df)
df2=drop_threshold(df,report,10)

print(check_missing(df2))

               column_name  total_missing  percentage_missing
0               Unnamed: 0              0                0.00
1              Clothing ID              0                0.00
2                      Age              0                0.00
3              Review Text            845                3.60
4                   Rating              0                0.00
5          Recommended IND              0                0.00
6  Positive Feedback Count              0                0.00
7            Division Name             14                0.06
8          Department Name             14                0.06
9               Class Name             14                0.06


In [8]:
result=cross_val_splits(df2)

In [21]:
def detect_column_types(df, unique_threshold=20, unique_ratio_threshold=0.05):
    result = {
        "numerical": [],
        "categorical": [],
        "datetime": [],
        "mixed": [],
        "encoded_categorical": []
    }

    for col in df.columns:
        s = df[col].dropna()
        
        if s.empty:
            result["mixed"].append(col)
            continue

        unique_count = s.nunique()
        unique_ratio = unique_count / len(s)

        # 1. datetime dtype
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            result["datetime"].append(col)
            continue

        # 2. numeric dtype
        if pd.api.types.is_numeric_dtype(df[col]):
            if unique_count <= unique_threshold or unique_ratio <= unique_ratio_threshold:
                result["encoded_categorical"].append(col)
            else:
                result["numerical"].append(col)
            continue

        # 3. object/string: try datetime
        datetime_converted = pd.to_datetime(s, errors="coerce")
        datetime_success_ratio = datetime_converted.notna().mean()

        if datetime_success_ratio > 0.8:
            result["datetime"].append(col)
            continue

        # 4. object/string: try numeric
        numeric_converted = pd.to_numeric(s, errors="coerce")
        numeric_success_ratio = numeric_converted.notna().mean()

        if numeric_success_ratio > 0.8:
            if unique_count <= unique_threshold or unique_ratio <= unique_ratio_threshold:
                result["encoded_categorical"].append(col)
            else:
                result["numerical"].append(col)
            continue

        # 5. mixed check
        has_numbers = numeric_converted.notna().any()
        has_text = numeric_converted.isna().any()

        if has_numbers and has_text:
            result["mixed"].append(col)
        else:
            result["categorical"].append(col)

    return result

In [22]:
data=pd.read_csv('data/house_prices_clean.csv')
print(data.head(),data.shape)
result=detect_column_types(data)
print(result)

       price  bedrooms  bathrooms  floors  waterfront  view  condition  \
0   313000.0       3.0       1.50     1.5           0     0          3   
1  2384000.0       5.0       2.50     2.0           0     4          5   
2   342000.0       3.0       2.00     1.0           0     0          4   
3   420000.0       3.0       2.25     1.0           0     0          4   
4   550000.0       4.0       2.50     1.0           0     0          4   

   yr_built  yr_renovated       location  footprint_ratio  basement_ratio  
0      1955          2005  Shoreline_133         0.169363        0.000000  
1      1921             0    Seattle_119         0.372376        0.076712  
2      1966             0       Kent_042         0.161547        0.000000  
3      1963             0   Bellevue_008         0.124533        0.500000  
4      1976          1992    Redmond_052         0.108571        0.412371   (4551, 12)
{'numerical': ['price', 'footprint_ratio', 'basement_ratio'], 'categorical': ['location'

C:\Users\Vineeth\AppData\Local\Temp\ipykernel_9344\199551216.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datetime_converted = pd.to_datetime(s, errors="coerce")


In [23]:
data=pd.read_csv('data/house_prices_clean.csv')
print(data.head(),data.shape)
result=detect_column_types(data,unique_threshold=10)
print(result)

       price  bedrooms  bathrooms  floors  waterfront  view  condition  \
0   313000.0       3.0       1.50     1.5           0     0          3   
1  2384000.0       5.0       2.50     2.0           0     4          5   
2   342000.0       3.0       2.00     1.0           0     0          4   
3   420000.0       3.0       2.25     1.0           0     0          4   
4   550000.0       4.0       2.50     1.0           0     0          4   

   yr_built  yr_renovated       location  footprint_ratio  basement_ratio  
0      1955          2005  Shoreline_133         0.169363        0.000000  
1      1921             0    Seattle_119         0.372376        0.076712  
2      1966             0       Kent_042         0.161547        0.000000  
3      1963             0   Bellevue_008         0.124533        0.500000  
4      1976          1992    Redmond_052         0.108571        0.412371   (4551, 12)
{'numerical': ['price', 'footprint_ratio', 'basement_ratio'], 'categorical': ['location'

C:\Users\Vineeth\AppData\Local\Temp\ipykernel_9344\199551216.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datetime_converted = pd.to_datetime(s, errors="coerce")


In [ ]:
def detect_column_types2(
    df,
    target: str = None,
    unique_threshold=20,
    unique_ratio_threshold=0.05
):
    result = {
        "numerical": [],
        "categorical": [],
        "datetime": [],
        "mixed": [],
        "encoded_categorical": [],
        "target": None
    }

    if target is not None:
        if not isinstance(target, str):
            raise TypeError("target must be a column name string, example: target='price'")

        if target not in df.columns:
            raise ValueError(f"Target column '{target}' not found in DataFrame.")

        target_series = df[target].dropna()

        if target_series.empty:
            raise ValueError(f"Target column '{target}' is empty.")

        if pd.api.types.is_numeric_dtype(target_series):
            result["target"] = {
                "name": target,
                "type": "numerical"
            }
        else:
            numeric_target = pd.to_numeric(target_series, errors="coerce")
            numeric_success_ratio = numeric_target.notna().mean()

            if numeric_success_ratio > 0.8:
                result["target"] = {
                    "name": target,
                    "type": "numerical"
                }
            else:
                raise ValueError(
                    f"Target column '{target}' looks categorical. "
                    "v1 is for regression only."
                )

    feature_df = df.drop(columns=[target]) if target is not None else df.copy()

    for col in feature_df.columns:
        s = feature_df[col].dropna()

        if s.empty:
            result["mixed"].append(col)
            continue

        unique_count = s.nunique()
        unique_ratio = unique_count / len(s)

        if pd.api.types.is_datetime64_any_dtype(feature_df[col]):
            result["datetime"].append(col)
            continue

        if pd.api.types.is_numeric_dtype(feature_df[col]):
            if unique_count <= unique_threshold or unique_ratio <= unique_ratio_threshold:
                result["encoded_categorical"].append(col)
            else:
                result["numerical"].append(col)
            continue

        datetime_converted = pd.to_datetime(s, errors="coerce")
        datetime_success_ratio = datetime_converted.notna().mean()

        if datetime_success_ratio > 0.8:
            result["datetime"].append(col)
            continue

        numeric_converted = pd.to_numeric(s, errors="coerce")
        numeric_success_ratio = numeric_converted.notna().mean()

        if numeric_success_ratio > 0.8:
            if unique_count <= unique_threshold or unique_ratio <= unique_ratio_threshold:
                result["encoded_categorical"].append(col)
            else:
                result["numerical"].append(col)
            continue

        has_numbers = numeric_converted.notna().any()
        has_text = numeric_converted.isna().any()

        if has_numbers and has_text:
            result["mixed"].append(col)
        else:
            result["categorical"].append(col)

    return result

In [32]:
data=pd.read_csv('data/house_prices_clean.csv')
print(data.head(),data.shape)
result=detect_column_types2(data,target='price')
print(result)

       price  bedrooms  bathrooms  floors  waterfront  view  condition  \
0   313000.0       3.0       1.50     1.5           0     0          3   
1  2384000.0       5.0       2.50     2.0           0     4          5   
2   342000.0       3.0       2.00     1.0           0     0          4   
3   420000.0       3.0       2.25     1.0           0     0          4   
4   550000.0       4.0       2.50     1.0           0     0          4   

   yr_built  yr_renovated       location  footprint_ratio  basement_ratio  
0      1955          2005  Shoreline_133         0.169363        0.000000  
1      1921             0    Seattle_119         0.372376        0.076712  
2      1966             0       Kent_042         0.161547        0.000000  
3      1963             0   Bellevue_008         0.124533        0.500000  
4      1976          1992    Redmond_052         0.108571        0.412371   (4551, 12)
{'numerical': ['footprint_ratio', 'basement_ratio'], 'categorical': ['location'], 'datet

C:\Users\Vineeth\AppData\Local\Temp\ipykernel_9344\1722077198.py:71: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datetime_converted = pd.to_datetime(s, errors="coerce")


In [33]:
data=pd.read_csv('data/house_prices_clean.csv')
print(data.head(),data.shape)
result=detect_column_types2(data,target='price')
print(result)

       price  bedrooms  bathrooms  floors  waterfront  view  condition  \
0   313000.0       3.0       1.50     1.5           0     0          3   
1  2384000.0       5.0       2.50     2.0           0     4          5   
2   342000.0       3.0       2.00     1.0           0     0          4   
3   420000.0       3.0       2.25     1.0           0     0          4   
4   550000.0       4.0       2.50     1.0           0     0          4   

   yr_built  yr_renovated       location  footprint_ratio  basement_ratio  
0      1955          2005  Shoreline_133         0.169363        0.000000  
1      1921             0    Seattle_119         0.372376        0.076712  
2      1966             0       Kent_042         0.161547        0.000000  
3      1963             0   Bellevue_008         0.124533        0.500000  
4      1976          1992    Redmond_052         0.108571        0.412371   (4551, 12)
{'numerical': ['footprint_ratio', 'basement_ratio'], 'categorical': ['location'], 'datet

C:\Users\Vineeth\AppData\Local\Temp\ipykernel_9344\1722077198.py:71: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datetime_converted = pd.to_datetime(s, errors="coerce")


In [36]:
df3=pd.read_csv('data/Titanic-Dataset.csv')
print(df3.head(),df3.shape)
result=detect_column_types2(df3,target='Survived')
print(result)

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S   (8

C:\Users\Vineeth\AppData\Local\Temp\ipykernel_9344\1722077198.py:71: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datetime_converted = pd.to_datetime(s, errors="coerce")
C:\Users\Vineeth\AppData\Local\Temp\ipykernel_9344\1722077198.py:71: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datetime_converted = pd.to_datetime(s, errors="coerce")
C:\Users\Vineeth\AppData\Local\Temp\ipykernel_9344\1722077198.py:71: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  datetime_converted = pd.to_datetime(s, errors="coerce")
C:\Users\Vineeth\AppData\Local\Temp\ipykernel_9344\1722077198.py:71

In [37]:
df3['Pclass'].head()

0    3
1    1
2    3
3    1
4    3
Name: Pclass, dtype: int64

In [ ]:
import pandas as pd


def group_rare_categories(df, columns, threshold, other_label="Other"):
    df = df.copy()

    for col in columns:
        freq = df[col].value_counts(normalize=True)
        rare_categories = freq[freq < threshold].index
        df[col] = df[col].apply(lambda x: other_label if x in rare_categories else x)

    return df